# 企业运营数据分析实战练习

请先运行“数据准备”代码单元，再按题目要求完成作答。

覆盖知识点：`read_csv`、`read_json`、`to_csv`、缺失值处理、`Timestamp`、`to_datetime`、`parse_dates`、日期索引、时间切片、时间差、`date_range`、`resample`、去重、类型转换、`category`、`map`、宽表长表转换、字符串拆分、`cut` / `qcut`、`rename`、`set_index`、`reset_index`、`groupby`、`agg`。


## 场景

你是某连锁零售企业的数据分析师，需要同时支持运营、HR、健康管理和天气关联分析。下面的数据均为教学用模拟数据，但题目设计尽量贴近实际工作场景。


In [34]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

required_files = [
    'sales_orders.csv',
    'employees.csv',
    'weather.csv',
    'weather_withna.csv',
    'users.json',
    'health_sleep.csv',
    'campaign_wide.csv',
]
missing_files = [name for name in required_files if not (DATA_DIR / name).exists()]
if missing_files:
    raise FileNotFoundError(f'缺少数据文件: {missing_files}')

# 以当前 data 目录中的文件为准做一次标准化，避免 notebook 内容与真实数据文件不一致。
sales_orders = pd.read_csv(DATA_DIR / 'sales_orders.csv')
sales_orders['discount'] = sales_orders['discount'].fillna(0)
sales_orders['revenue'] = sales_orders['quantity'] * sales_orders['unit_price'] - sales_orders['discount']
sales_orders.to_csv(DATA_DIR / 'sales_orders.csv', index=False, encoding='utf-8-sig')

employees = pd.read_csv(DATA_DIR / 'employees.csv')
employees.to_csv(DATA_DIR / 'employees.csv', index=False, encoding='utf-8-sig')

weather = pd.read_csv(DATA_DIR / 'weather.csv')
weather.to_csv(DATA_DIR / 'weather.csv', index=False, encoding='utf-8-sig')

weather_withna = pd.read_csv(DATA_DIR / 'weather_withna.csv')
weather_withna.to_csv(DATA_DIR / 'weather_withna.csv', index=False, encoding='utf-8-sig')

with open(DATA_DIR / 'users.json', 'r', encoding='utf-8') as f:
    users = json.load(f)
with open(DATA_DIR / 'users.json', 'w', encoding='utf-8') as f:
    json.dump(users, f, ensure_ascii=False, indent=2)

health = pd.read_csv(DATA_DIR / 'health_sleep.csv')
health.to_csv(DATA_DIR / 'health_sleep.csv', index=False, encoding='utf-8-sig')

campaign_wide = pd.read_csv(DATA_DIR / 'campaign_wide.csv')
campaign_wide.to_csv(DATA_DIR / 'campaign_wide.csv', index=False, encoding='utf-8-sig')

print('数据校准完成：', DATA_DIR.resolve())
print({name: (DATA_DIR / name).stat().st_size for name in required_files})


数据校准完成： E:\Projects\DataProcessing\04_数据分析\data
{'sales_orders.csv': 1130, 'employees.csv': 387, 'weather.csv': 3454, 'weather_withna.csv': 3454, 'users.json': 537, 'health_sleep.csv': 367, 'campaign_wide.csv': 163}


## 第 1 题：销售订单数据导入与导出

业务要求：运营经理要快速查看订单数据的基础情况。

1. 读取 `data/sales_orders.csv`。
2. 查看数据类型、尾部 5 行。
3. 计算 `revenue` 的平均值。
4. 取 `revenue` 最高的前 6 条订单，导出到 `output/top6_orders.csv`。


In [35]:
# 提示：可先创建 output 目录
OUTPUT_DIR = Path('output')
# exist_ok=True：如果目录存在不会抛出FileExistsError错误
OUTPUT_DIR.mkdir(exist_ok=True)
# 读取csv文件
sales_orders = pd.read_csv(DATA_DIR / 'sales_orders.csv')
# 查看数据类型、尾部5行
print(sales_orders.dtypes)
print(sales_orders.tail(5))
# 计算营业额的平均值
# 先处理discount缺失数据
sales_orders['discount'].fillna(0, inplace=True)
sales_orders['revenue'] = sales_orders['quantity'] * sales_orders['unit_price'] - sales_orders['discount']
print("revenue的平均值 = ", sales_orders['revenue'].mean())
# 取revenue最高的前六条订单
sales_orders.sort_values('revenue', ascending=False).head(6).to_csv(OUTPUT_DIR / 'top6_orders.csv', index=False, encoding='utf-8-sig')



order_id          int64
order_date       object
city             object
channel          object
category         object
quantity          int64
unit_price        int64
discount        float64
order_status     object
revenue         float64
dtype: object
    order_id  order_date city channel category  quantity  unit_price  \
11      1012  2025-04-02   广州      门店       食品        12          32   
12      1013  2025-04-09   深圳     App       家电         1        2899   
13      1014  2025-04-16   杭州      直播       美妆         4         219   
14      1015  2025-04-21   上海     小程序       日用         5          99   
15      1015  2025-04-21   上海     小程序       日用         5          99   

    discount order_status  revenue  
11       0.0          已完成    384.0  
12     100.0          已完成   2799.0  
13      20.0          已完成    856.0  
14       0.0          已完成    495.0  
15       0.0          已完成    495.0  
revenue的平均值 =  1789.5625


## 第 2 题：JSON 用户档案整理

业务要求：CRM（客户关系团队） 团队提供了一份 JSON 格式的用户档案。

1. 读取 `data/users.json`。
2. 将 `users` 节点转换为 DataFrame。
3. 输出 DataFrame 的前 3 行，并说明其数据结构类型。


In [36]:
# 读取data/users.json
with open(DATA_DIR/'users.json') as f:
    users = json.load(f)
# users阶段转换为DataFrame
users_df = pd.DataFrame(users['users'])
# 输出前3行并说明数据结构类型
print(users_df.head(3), '\n', type(users_df.head(3)))


   user_id    full_name             email  gender
0        1  alice zhang    ALICE@demo.com  Female
1        2       bob li    bob@company.cn    Male
2        3   cathy wang  Cathy@school.edu  Female 
 <class 'pandas.core.frame.DataFrame'>


## 第 3 题：天气数据缺失值处理

业务要求：门店客流分析要关联天气，但天气表中有缺失值。

1. 读取 `data/weather_withna.csv`。
2. 统计每列缺失值数量。
3. 删除 `temp_max` 缺失的记录。
4. 使用字典填充 `wind` 和 `precipitation` 的缺失值。
5. 再分别演示一次使用均值填充、前向填充、后向填充。


In [37]:
# 读取天气数据
weather_df = pd.read_csv(DATA_DIR / 'weather_withna.csv')
# 统计每列缺失值数量
print(weather_df.isna().sum())
# 删除temp_max缺失的记录
weather_df.dropna(subset='temp_max', inplace=True)
# 使用字典填充wind和precipitation的缺失值
# wind_avg = round(weather_df['wind'].mean(), 1)
# precipitation_avg = round(weather_df['precipitation'].mean(), 1)
# weather_df.fillna({'wind': wind_avg, "precipitation": precipitation_avg}, inplace=True)
# 均值填充
# weather_df.fillna(round(weather_df[['wind', 'precipitation']].mean(), 1), inplace=True)
# 前向填充
# weather_df.ffill(inplace=True)
# 后向填充
weather_df.bfill(inplace=True)

date              0
temp_max          3
temp_min          0
precipitation    14
wind             23
dtype: int64


## 第 4 题：报表时间戳处理

业务要求：你需要生成一个“报表生成时间”对象，并从中抽取常用时间维度。

1. 使用 `pd.Timestamp` 创建 `2025-04-30 18:45`。
2. 提取年、月、日、小时、季度。
3. 判断这一天是否月底。
4. 将该时间分别转换为日、月、季度周期。


In [38]:
# 创建时间戳
data = (pd.Timestamp('2025-04-30 18:45'))
# 提取年、月、日、小时、季度
print(data.year, data.month, data.day, data.hour, data.quarter)
# 判断是否为月底
print("是月底" if data.is_month_end else "不是月底")
# 日期转换
print(data.to_period('D'))
print(data.to_period('M'))
print(data.to_period('Q'))


2025 4 30 18 2
是月底
2025-04-30
2025-04
2025Q2


## 第 5 题：订单日期字段转换

业务要求：销售订单的日期目前是字符串，无法直接用于时间分析。

1. 读取 `data/sales_orders.csv`。
2. 将 `order_date` 转为日期类型。
3. 新增 `weekday`、`month_period` 两列。
4. 输出转换后的数据类型信息。


In [39]:
# 读取"data/sales/orders.csv"
sales_orders = pd.read_csv(DATA_DIR / 'sales_orders.csv')
sales_orders.head()
# 将order_date转为日期类型
sales_orders['order_date'] = pd.to_datetime(sales_orders['order_date'])
# 新增weekday（星期）, month_period（季度）两列
sales_orders['weekday'] = sales_orders['order_date'].dt.day_name()
sales_orders['month_period'] = sales_orders['order_date'].dt.to_period('M')
# 输出转换后数据类型信息
print(sales_orders.dtypes)

order_id                 int64
order_date      datetime64[ns]
city                    object
channel                 object
category                object
quantity                 int64
unit_price               int64
discount               float64
order_status            object
revenue                float64
weekday                 object
month_period         period[M]
dtype: object


## 第 6 题：日期索引、时间切片与时间差

业务要求：市场部要查看一段时间内的天气变化。

1. 读取 `data/weather.csv`，使用 `parse_dates` 解析 `date`。
2. 将 `date` 设为索引。
3. 切片查询 `2025-02-01` 到 `2025-02-15` 的记录。
4. 新增一列 `delta`，表示距离首日的时间差。
5. 再把 `delta` 设为索引，筛选 `10 days` 到 `20 days` 的记录。


In [40]:
# 读取 `data/weather.csv`，使用 `parse_dates` 解析 `date`。
weather_df = pd.read_csv(DATA_DIR/'weather.csv', parse_dates=['date'])
# 将 `date` 设为索引。
weather_df.set_index('date', inplace=True)
# 切片查询 `2025-02-01` 到 `2025-02-15` 的记录。
print(weather_df.loc['2025-02-01': '2025-02-15'])
# 新增一列 `delta`，表示距离首日的时间差。
weather_df['delta'] = weather_df.index - weather_df.index[0]
# 再把 `delta` 设为索引，筛选 `10 days` 到 `20 days` 的记录。
weather_df.set_index('delta', inplace=True)
print(weather_df.loc['10 days': '20 days'])

            temp_max  temp_min  precipitation  wind
date                                               
2025-02-01       3.0      23.0            0.0   1.8
2025-02-02      14.0       1.0            1.5   4.1
2025-02-03      28.0       6.0            8.8   1.8
2025-02-04      24.0       3.0            NaN   NaN
2025-02-05      31.0      22.0            3.2   4.1
2025-02-06      14.0       5.0            0.0   NaN
2025-02-07       NaN      24.0            0.0   3.2
2025-02-08      19.0       3.0            0.2   4.1
2025-02-09      29.0       5.0            8.8   3.2
2025-02-10      29.0       4.0            0.0   1.8
2025-02-11      33.0       8.0            3.2   1.8
2025-02-12      12.0      -2.0            1.5   4.1
2025-02-13      34.0      -3.0            0.0   4.1
2025-02-14      30.0      12.0            8.8   NaN
2025-02-15      30.0      19.0            NaN   NaN
         temp_max  temp_min  precipitation  wind
delta                                           
10 days      21.0 

## 第 7 题：日期序列与重采样

业务要求：老板希望同时看到“周报日历”和“月度天气均值”。

1. 使用 `pd.date_range` 生成从 `2025-01-05` 开始、连续 8 个周日的日期序列。
2. 读取 `data/weather.csv` 并将 `date` 设为索引。
3. 对 `temp_max`、`temp_min` 做月初频率 (`MS`) 的均值重采样。
4. 对相同字段再做年末频率 (`YE`) 的均值重采样。


In [41]:
# 生成从 2025-01-05 开始、连续 8 个周日的日期序列。
eight_sundays = pd.date_range('2025-01-05', periods=8, freq='W')
print(eight_sundays)
# 读取 weather.csv，并把 date 解析为日期索引。
weather_df = pd.read_csv(DATA_DIR/'weather.csv', parse_dates=['date'], index_col='date')
# 按月初频率（MS）计算 temp_max 和 temp_min 的均值。
print(weather_df[['temp_max', 'temp_min']].resample('MS').mean())
# 当前 pandas 1.4.4 环境中可以使用 Y，写 YE 会报 Invalid frequency。
print(weather_df[['temp_max', 'temp_min']].resample('Y').mean())


DatetimeIndex(['2025-01-05', '2025-01-12', '2025-01-19', '2025-01-26',
               '2025-02-02', '2025-02-09', '2025-02-16', '2025-02-23'],
              dtype='datetime64[ns]', freq='W-SUN')
             temp_max   temp_min
date                            
2025-01-01  18.862069  11.000000
2025-02-01  23.185185   8.178571
2025-03-01  17.387097   9.806452
2025-04-01  18.766667  12.566667
             temp_max  temp_min
date                           
2025-12-31  19.444444    10.425


## 第 8 题：重复值检查与去重

业务要求：订单表和员工表中都可能出现重复记录，影响统计结果。

1. 读取 `data/sales_orders.csv`，检查是否存在重复订单。
2. 按整行去重后输出结果行数。
3. 读取 `data/employees.csv`，按 `employee_id` 去重，保留最后一次出现的记录。


In [42]:
# 读取 `data/sales_orders.csv`，检查是否存在重复订单。
sales_orders = pd.read_csv(DATA_DIR/'sales_orders.csv')
print(sales_orders.duplicated())
# 按整行去重后输出结果行数。
sales_orders.drop_duplicates(inplace=True)
print(len(sales_orders))
# 读取 `data/employees.csv`，按 `employee_id` 去重，保留最后一次出现的#
employees_df =pd.read_csv(DATA_DIR/'employees.csv')
employees_df.drop_duplicates(subset=['employee_id'], keep='last', inplace=True)
print(employees_df)

0     False
1     False
2     False
3     False
4     False
5     False
6     False
7     False
8     False
9     False
10    False
11    False
12    False
13    False
14    False
15     True
dtype: bool
15
   employee_id region  department_id   job_id  salary   join_date
0          201     华东           10.0   SA_MAN   26000  2021-03-15
1          202     华北           20.0  IT_PROG   18000  2022-07-01
2          203     华南           20.0  IT_PROG   19500  2023-01-10
3          204     华东           30.0   MK_REP   15000  2024-05-18
4          205     华北            NaN   SA_REP   12000  2024-07-02
5          206     华南           10.0   SA_REP   13500  2023-10-11
7          207     华东           40.0   HR_REP   12500  2022-12-05


## 第 9 题：数据类型转换与映射

业务要求：HR 分析前需要规范字段类型。

1. 读取 `data/employees.csv`。
2. 删除 `department_id` 为空的记录。
3. 将 `department_id` 转为 `int64`。
4. 读取 `data/users.json` 中的用户表，将 `gender` 转为 `category`。
5. 新增一列 `is_male`，使用 `map` 映射为布尔值。


In [43]:
# 读取 `data/employees.csv`。
employees_df = pd.read_csv(DATA_DIR/'employees.csv')
# print(employees_df.dtypes)
# 删除 `department_id` 为空的记录。
employees_df.dropna(subset=['department_id'], inplace=True)
# 将 `department_id` 转为 `int64`。
employees_df['department_id'] = employees_df['department_id'].astype('int64')
# 读取 `data/users.json` 中的用户表，将 `gender` 转为 `category`。
with open(DATA_DIR/'users.json') as f:
    users = json.load(f)
users_df = pd.DataFrame(users['users'])
users_df['gender'] = users_df['gender'].astype('category')
print(users_df.dtypes)
# 新增一列 `is_male`，使用 `map` 映射为布尔值。
users_df['is_male'] = users_df['gender'].map({'Male':True, 'Female':False})
print(users_df)

user_id         int64
full_name      object
email          object
gender       category
dtype: object
   user_id    full_name             email  gender is_male
0        1  alice zhang    ALICE@demo.com  Female   False
1        2       bob li    bob@company.cn    Male    True
2        3   cathy wang  Cathy@school.edu  Female   False
3        4   david chen    david@demo.com    Male    True


## 第 10 题：宽表与长表转换

业务要求：营销部给的是宽表，但 BI 系统需要长表。

1. 读取 `data/campaign_wide.csv`。
2. 将 `曝光量`、`点击量`、`成交量` 由宽表转成长表。
3. 长表转换后，类别列命名为 `指标`，数值列命名为 `指标值`。
4. 按 `campaign_name` 排序查看结果。
5. 再将长表转回宽表。


In [44]:
# 读取 `data/campaign_wide.csv`。
campaign_wide = pd.read_csv(DATA_DIR/'campaign_wide.csv')
# 将 `曝光量`、`点击量`、`成交量` 由宽表转成长表。
campaign_long = campaign_wide.melt(id_vars=['campaign_id','campaign_name'], value_vars=['曝光量','点击量','成交量'], var_name='指标', value_name='指标值')
# 按 `campaign_name` 排序查看结果。
print(campaign_long.sort_values('campaign_name'))
# 长表转成宽表
campaign_wide_again = campaign_long.pivot(index=['campaign_id','campaign_name'], columns='指标', values='指标值').reset_index()
print(campaign_wide_again)


   campaign_id campaign_name   指标     指标值
2         9003          会员复购  曝光量   86000
5         9003          会员复购  点击量    6400
8         9003          会员复购  成交量     760
1         9002          春季上新  曝光量   98000
4         9002          春季上新  点击量    7200
7         9002          春季上新  成交量     810
0         9001          春节拉新  曝光量  120000
3         9001          春节拉新  点击量    8600
6         9001          春节拉新  成交量     920
指标  campaign_id campaign_name  成交量     曝光量   点击量
0          9001          春节拉新  920  120000  8600
1          9002          春季上新  810   98000  7200
2          9003          会员复购  760   86000  6400


## 第 11 题：字符串拆分与健康字段加工

业务要求：健康管理中心要对姓名、邮箱和血压字段进行拆分。

1. 读取用户 JSON 转成 DataFrame。
2. 将 `full_name` 拆成 `first_name` 和 `last_name`。
3. 将 `first_name` 处理为首字母大写。
4. 提取邮箱域名到 `email_domain`，并新增 `邮箱类型`（`.com` 为 `商业`，否则为 `其他`）。
5. 读取 `data/health_sleep.csv`，保留 `person_id` 和 `blood_pressure`。
6. 将血压拆分为 `high`、`low`，并转成整数类型。


In [45]:
# 读取用户 JSON 转成 DataFrame。
with open(DATA_DIR/'users.json') as f:
    users_map = json.load(f)
users_df = pd.DataFrame(users_map['users'])
# 将 `full_name` 拆成 `first_name` 和 `last_name`。
users_df[['first_name', 'last_name']] = users_df['full_name'].str.split(' ', expand=True)
# 将 `first_name` 处理为首字母大写。
users_df['first_name'] = users_df['first_name'].str.capitalize()
# 提取邮箱域名到 `email_domain`，并新增 `邮箱类型`（`.com` 为 `商业`，否则为 `其他`）。
users_df['email_domain'] = users_df['email'].str.partition('@')[2].str.lower()
users_df['邮箱类型'] = np.where(users_df['email_domain'].str.endswith('.com'), '商业', '其他')
# 读取 `data/health_sleep.csv`，保留 `person_id` 和 `blood_pressure`。
sleep_df = pd.read_csv(DATA_DIR/'health_sleep.csv', usecols=['person_id', 'blood_pressure'])
# 将血压拆分为 `high`、`low`，并转成整数#
sleep_df[['high', 'low']] = sleep_df['blood_pressure'].str.split('/', expand=True)
sleep_df['high'] = sleep_df['high'].astype('int64')
sleep_df['low'] = sleep_df['low'].astype('int64')

## 第 12 题：数据分箱

业务要求：为了方便汇报，需要把订单金额和睡眠质量分成等级。

1. 基于订单表，计算每笔订单的 `order_amount`（可直接用 `revenue`）。
2. 使用 `pd.cut` 将订单金额按 `[0, 500, 3000, 8000]` 分为 `低`、`中`、`高`。
3. 使用 `pd.qcut` 将订单金额等频分成 3 档，并统计每档数量。
4. 基于 `data/health_sleep.csv`，将 `sleep_quality` 分成 `差`、`中`、`优` 三档。


In [46]:
# 基于订单表，计算每笔订单的 `order_amount`（可直接用 `revenue`）。
# 由于有revenue了，所以直接略过
# 使用 `pd.cut` 将订单金额按 `[0, 500, 3000, 8000]` 分为 `低`、`中`、`高`。
sales_orders['订单等级'] = pd.cut(sales_orders['revenue'], bins=[0, 500, 3000, 8000], labels=['低', '中', '高'])
# 使用 `pd.qcut` 将订单金额等频分成 3 档，并统计每档数量。
print(pd.qcut(sales_orders['revenue'], q=3).value_counts())
# 基于 `data/health_sleep.csv`，将 `sleep_quality` 分成 `差`、`中`、`优` 三档。
sleep_df = pd.read_csv(DATA_DIR/'health_sleep.csv', usecols=['person_id', 'sleep_quality'])
sleep_df['睡眠质量'] = pd.cut(sleep_df['sleep_quality'], bins=3, labels=['差', '中', '优'])


(274.999, 488.0]      5
(488.0, 2598.333]     5
(2598.333, 6699.0]    5
Name: revenue, dtype: int64


## 第 13 题：列名重命名、索引设置与还原

业务要求：周报导出前，需要统一字段命名。

1. 读取 `data/employees.csv` 的前 5 行。
2. 将 `employee_id` 重命名为 `员工ID`，`salary` 重命名为 `月薪`。
3. 将 `员工ID` 设为索引。
4. 再将索引还原成普通列。


In [47]:
# 读取 `data/employees.csv` 的前 5 行。
employees_df = pd.read_csv(DATA_DIR/'employees.csv').head(5)
# 将 `employee_id` 重命名为 `员工ID`，`salary` 重命名为 `月薪`。
employees_df.rename(columns={'employee_id':'员工ID','salary':'月薪'}, inplace=True)
# 将 `员工ID` 设为索引。
employees_df.set_index('员工ID', inplace=True)
# 再将索引还原成普通列。
employees_df.reset_index(inplace=True)

## 第 14 题：分组聚合分析

业务要求：管理层要看部门薪资和渠道销售表现。

1. 对员工表按 `department_id` 统计平均薪资，并按薪资降序排序。
2. 对员工表按 `department_id`、`job_id` 统计平均薪资。
3. 对订单表按 `city`、`channel` 分组，统计：
   - `revenue` 总和
   - `order_id` 数量
   - `quantity` 平均值
4. 将结果整理成适合汇报的 DataFrame。


In [48]:
# 按 department_id 统计平均薪资，并按降序排序。
employees_df = pd.read_csv(DATA_DIR/'employees.csv')
dep_aversalary = employees_df.groupby('department_id')[['salary']].mean().sort_values('salary', ascending=False)
print(dep_aversalary)
# 按 department_id 和 job_id 统计平均薪资。
dep_job_aversalary = employees_df.groupby(['department_id','job_id'])[['salary']].mean().sort_values('salary', ascending=False)
print(dep_job_aversalary)
# 订单表先按整行去重，再按 city 和 channel 分组汇总。
sales_orders = pd.read_csv(DATA_DIR / 'sales_orders.csv').drop_duplicates()
sales_orders_group = sales_orders.groupby(['city', 'channel']).agg(
    revenue=('revenue', 'sum'),
    order_count=('order_id', 'nunique'),
    avg_quantity=('quantity', 'mean')
).reset_index()
print(sales_orders_group)


                salary
department_id         
10.0           19750.0
20.0           18750.0
30.0           15000.0
40.0           12500.0
                        salary
department_id job_id          
10.0          SA_MAN   26000.0
20.0          IT_PROG  18750.0
30.0          MK_REP   15000.0
10.0          SA_REP   13500.0
40.0          HR_REP   12500.0
   city channel  revenue  order_count  avg_quantity
0    上海     App   6699.0            1           1.0
1    上海     小程序    495.0            1           5.0
2    上海      直播   5646.0            2           2.0
3    北京     App   4273.0            2           3.5
4    北京     小程序   2598.0            1           2.0
5    广州      门店   2983.0            2           6.5
6    杭州     App    888.0            1           2.0
7    杭州      直播    856.0            1           4.0
8    杭州      门店    280.0            1           8.0
9    深圳     App   3145.0            2           2.5
10   深圳     小程序    275.0            1          10.0


## 第 15 题：综合业务分析小结

业务要求：请基于前面的处理结果，完成一段 5 行以内的分析结论，至少包含以下信息：

1. 哪个城市或渠道贡献最高。
2. 哪个部门平均薪资最高。
3. 天气数据和订单分析后，你认为还可以继续补充哪些字段。

本题写在 Markdown 中即可。


1. 按城市汇总，上海的贡献最高；按渠道汇总，App 的贡献最高。
2. 按当前 employees.csv 的计算结果，department_id 为 10 的平均薪资最高。
3. 天气数据还可以补充平均气温、降水等级、风力等级；订单数据还可以补充成本价、毛利、净利润、退货原因等字段。
